In [ ]:
# Auto-reload modules during development
%load_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt

# Import the framework
from nowcasting_framework import (
    NowcastConfig,
    DataManager,
    ModelManager,
    EvaluationManager,
    VisualizationManager,
    InferencePipeline
)

# Import model
from sklearn.ensemble import GradientBoostingRegressor

## 1. Configuration

In [ ]:
# Configure Gradient Boosting experiment
# Using 6 lags and 10-model ensemble
config = NowcastConfig(
    target_variable="gdpc1",
    test_start_date="2005-03-01",
    test_end_date="2010-03-01",
    n_lags=6,  # More lags for gradient boosting
    n_ensemble_models=10,  # Average 10 models due to stochasticity
    quarterly_only=True
)

print("Gradient Boosting Configuration:")
print("=" * 60)
for key, value in config.to_dict().items():
    print(f"{key:20s}: {value}")
print("=" * 60)

## 2. Data Loading

In [ ]:
# Initialize and load data
data_manager = DataManager(config)
data_manager.load_data().prepare_test_data()

# Display data summary
data_manager.summary()

## 3. Model Training and Backtesting

In [ ]:
# Configure Gradient Boosting parameters
gb_params = {
    'loss': 'absolute_error',
    'learning_rate': 0.1,
    'n_estimators': 100,
    'max_depth': 3,
    'random_state': 42
}

# Create model manager and run backtest
gb_model = ModelManager(GradientBoostingRegressor, gb_params, config)
gb_model.run_backtest(data_manager)

## 4. Evaluation

In [ ]:
# Evaluate model performance
evaluator = EvaluationManager()
evaluator.add_model_results(
    'Gradient Boosting (10-ensemble)',
    gb_model.get_predictions(),
    data_manager.actuals,
    config.lags
)

# Display performance metrics
print("\nPERFORMANCE BY VINTAGE:")
print("-" * 60)
display(evaluator.get_performance_table().round(6))

print("\n" + evaluator.summary_report())

## 5. Visualization

In [ ]:
# Create visualization manager
viz = VisualizationManager(evaluator)

# Plot predictions vs actuals
fig1 = viz.plot_predictions_vs_actuals('Gradient Boosting (10-ensemble)')
plt.show()

In [ ]:
# Plot error distribution
fig2 = viz.plot_error_distribution('Gradient Boosting (10-ensemble)')
plt.show()

## 6. Production Inference

In [ ]:
# Create inference pipeline
inference = InferencePipeline(gb_model, data_manager, config)

# Predict future quarter
prediction = inference.predict_new_date("2010-06-01")

print("\nFUTURE PREDICTION:")
print("=" * 60)
for key, value in prediction.items():
    print(f"{key:20s}: {value}")

## Summary

This notebook implements Gradient Boosting regression from scikit-learn for GDP nowcasting. Like XGBoost, this builds an ensemble of trees sequentially, but with different algorithmic choices. We use absolute error loss for robustness and average predictions from 10 models.

With 6 lags and 100 estimators, this configuration balances model complexity with computational efficiency. The framework approach handles the backtesting mechanics, making it straightforward to compare this against other boosting and ensemble methods.